In [4]:
%load_ext autoreload
%autoreload 2
import pandas as pd
import regex
import clinlp

import os
from tqdm import tqdm
import numpy as np
import clinlp

import spacy


import pyreadstat

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [5]:
# perotinitis/lijnsepsis; complicaties dialyse; uit consult teksten

# Einddatum dialyse; consult teksten/ontslag

# vorm van dialyse; uit consult teksten

# functionerende shunt

# type vaattoegang

# dialyse dosis

# rejectie; patho

In [7]:
os.chdir('/Volumes/DATA/LAB/lab_research/RES-Folder-UPOD/Coforix')

FileNotFoundError: [Errno 2] No such file or directory: '/Volumes/DATA/LAB/lab_research/RES-Folder-UPOD/Coforix'

In [ ]:
file_list = os.listdir('E_ResearchData/2_ResearchData/20260928_Bram')
file_list = [f for f in file_list if f.endswith('.parquet')]
file_list = sorted( file_list,
                        key =  lambda x: os.stat
                       (os.path.join('E_ResearchData/2_ResearchData/20260928_Bram', x)).st_size)

In [ ]:
coforix_data = {}
for file_name in tqdm(file_list):
    small_file = os.path.join('E_ResearchData/2_ResearchData/20260928_Bram', file_name)
    try:
        coforix_data[file_name.strip('.parquet')] = pd.read_parquet(small_file, encoding='latin-1', compression='infer')
    except Exception as e:
        print(f"Failed to read {small_file}: {e}")


100%|██████████| 3/3 [32:32<00:00, 650.82s/it]


In [ ]:
# # write to parquet 
# for file_name, df in coforix_data.items():
#     parquet_file = os.path.join('E_ResearchData/2_ResearchData/20260928_Bram', f"{file_name}.parquet")
#     df.to_parquet(parquet_file, index=False)

In [7]:
coforix_data.keys()

dict_keys(['nc_documenten_20260928', 'nc_consult_20260928', 'nc_vragenlijst_20260928'])

In [ ]:
coforix_data['nc_consult_20260928'] = coforix_data['nc_consult_20260928'].dropna(subset=['section_text'], axis=0)
coforix_data['nc_documenten_20260928'] = coforix_data['nc_documenten_20260928'].dropna(subset=['doc_tekst_plain'], axis=0)

In [42]:
np.median(coforix_data['nc_documenten_20260928'].groupby('studyId_1018').size())

np.float64(4.0)

In [43]:
np.median(coforix_data['nc_consult_20260928'].groupby('studyId_1018').size())

np.float64(31.0)

In [44]:
np.median(coforix_data['nc_vragenlijst_20260928'].groupby('studyId_1018').size())

np.float64(205.0)

# Omschrijving nierziekte
* hoeveel peritonitis per 1000 katheterdagen
* hoeveel lijnsepsis per 1000 katheterdagen


Peritonitis: peritonitis, buikvliesontsteking, PD-peritonitis, CAPD-peritonitis, dialyseperitonitis

Lijnsepsis: lijnsepsis, lijninfectie, kathetersepsis, kathetergerelateerde sepsis, CRBSI, kathetergerelateerde bacteriëmie, dialysekatheterinfectie

In [63]:
nlp = spacy.blank("clinlp")

# Normalization
nlp.add_pipe("clinlp_normalizer")

# Sentences
nlp.add_pipe("clinlp_sentencizer")

# Entities
terms = {
    "peritonitis": [
        "peritonitis", 
        "buikvliesontsteking",
        clinlp.ie.Term("dialyseperitonitis", proximity=1, fuzzy=True, case_sensitive=False),
        clinlp.ie.Term("peritoneale ontsteking", proximity=1, case_sensitive=False), 
        clinlp.ie.Term("peritonitis bacteriana", proximity=1, case_sensitive=False),
        clinlp.ie.Term("peritonitis acuta", proximity=1, case_sensitive=False),
        clinlp.ie.Term("PD-peritonitis", proximity=1, case_sensitive=False),
        clinlp.ie.Term("CAPD-peritonitis", proximity=1, case_sensitive=False),
        clinlp.ie.Term("dialyse peritonitis", proximity=1, fuzzy=True, case_sensitive=False),
        
    ],
    "lijn_sepsis": [
        "lijninfectie", 
        "kathetersepsis",
        "CRBSI",
        "dialysekatheterinfectie",
        clinlp.ie.Term("kathetergerelateerde bacteriëmie", proximity=1, fuzzy=True, case_sensitive=False), 
        clinlp.ie.Term("lijn sepsis", proximity=1, case_sensitive=False),
        clinlp.ie.Term("dialyse katheterinfectie", proximity=1, case_sensitive=False)
    ],
}


In [51]:
entity_matcher = nlp.add_pipe("clinlp_rule_based_entity_matcher", config={"attr": "NORM", "fuzzy": 1})
entity_matcher.add_terms_from_dict(terms)
# Qualifiers
nlp.add_pipe("clinlp_context_algorithm", config={"phrase_matcher_attr": "NORM"})

# text = (
#     "Preterme neonaat (<p3) opgenomen, bd enigszins verlaagd, familieanamnese vermeldt eveneens hypotensie "
#     "bij moeder. Thans geen aanwijzingen voor veneus infarkt wat ook geen "
#     "verklaring voor de partus prematurus is. Risico op VI blijft aanwezig."
# )

# doc = nlp(text)

In [54]:
docs = [nlp(txt) for txt in coforix_data['nc_consult_20260928'].section_text]

In [60]:
ents_ = [len(doc.ents) for doc in docs]

In [ ]:
coforix_data['nc_consult_20260928'].section_text.str.extract(r'(peritonitis|lijn sepsis)', regex=True)

# Medication

medicat_medicijn, medicat_schem, medicat_issuereque, medicat_recdeel,zindex_801, zindex_711, zindex_050, zindex_031, zindex_004:
look for immunosuppressors, pain killers, when start/stop

Solumedrol,Tacrolimus



# Text 

wi_docascii -> PLATTETEXT column, DOCID, DOCID to patient via wi_documen

wi_docascii: 
throw out: "Dit  bericht is automatisch gegenereerd..."


consult_regpar -> TEXT,plain_text, REGISTRATIONID,PARTID, to patient via patient_preferre->consult_regmodif

questresp_item_hixddrantw

consult_regpar:
remove RTF shit

* look for "Nefrologie"

* look for "i.v.m. transplantatie" , "Eurotransplant", "registratie bij Eurotransplant actief"
* look for "Pre-transplantatie"
* look for "Post-transplantatie"
* look for "therapie" + "Nefrologie"
* look for "Nierziekte" 
* look for "Nierinsufficiëntie"
* look for "Chronische nierinsufficiëntie"
* look for "nefrectomie" 
* look for "niercelcarcinoom"
* look for "eGFR\s?[0-9]{1,3}
* look for "Immuunsupressie", "Tacrolimus", "rejectie", "afwijzing"
* look for "ADPKD"


# medisch algemeen
* hoeveel dagen opnames per specialisme per jaar


# Omschrijving transplantatie
* afstoting
* pre-emptieve transplantatie
* overleving


# extraheer quantiteit en eenheid
lab_huidige_uitslag
* Albumine
* Creatinine
* eiwit
* serum fosfaat
* serum hb
* serum bicarbonaat






# 